In [0]:
from datetime import datetime

dbutils.widgets.text("batch_date", datetime.now().strftime("%Y-%m-%d"))
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.dropdown("entity", "all", ["all", "commits", "issues", "pulls"])
dbutils.widgets.text("raw_root", "/Volumes/workspace/landing/raw")

args = {k: dbutils.widgets.get(k) for k in ["batch_date", "load_type", "entity", "raw_root"]}
run_start = datetime.now()

def failures():
    return spark.sql(f"""SELECT layer, entity, error_message FROM workspace.ops.pipeline_execution_logs
        WHERE status = 'FAILURE' AND start_time >= TIMESTAMP '{run_start:%Y-%m-%d %H:%M:%S}'
        AND parameter LIKE '{args["load_type"]}/{args["batch_date"]}/%'""").collect()

steps = ["02_raw_to_bronze", "03_bronze_to_silver", "04_silver_to_gold"]
for nb in steps:
    t0 = datetime.now()
    print(f"== running {nb} with {args}")
    dbutils.notebook.run(f"./{nb}", 3600, args)
    bad = failures()
    if bad:
        raise Exception(f"{nb} had failures: {bad}")
    print(f"   {nb} done in {(datetime.now() - t0).seconds}s")

print("Pipeline finished OK: Raw -> Bronze -> Silver -> Gold")

== running 02_raw_to_bronze with {'batch_date': '2026-10-19', 'load_type': 'incremental', 'entity': 'all', 'raw_root': '/Volumes/workspace/landing/raw'}
   02_raw_to_bronze done in 143s
== running 03_bronze_to_silver with {'batch_date': '2026-10-19', 'load_type': 'incremental', 'entity': 'all', 'raw_root': '/Volumes/workspace/landing/raw'}
   03_bronze_to_silver done in 62s
== running 04_silver_to_gold with {'batch_date': '2026-10-19', 'load_type': 'incremental', 'entity': 'all', 'raw_root': '/Volumes/workspace/landing/raw'}
   04_silver_to_gold done in 143s
Pipeline finished OK: Raw -> Bronze -> Silver -> Gold


In [0]:
%sql
SELECT layer, entity, rows_inserted, rows_updated,
       round(unix_timestamp(end_time) - unix_timestamp(start_time), 1) AS seconds
FROM workspace.ops.pipeline_execution_logs
WHERE parameter LIKE 'full/2026-10-01/%'
ORDER BY start_time DESC LIMIT 14   

layer,entity,rows_inserted,rows_updated,seconds
silver_to_gold,gold_repo_summary,4,0,3
silver_to_gold,gold_issue_pr_turnaround,84,0,4
silver_to_gold,gold_repo_momentum,85,0,4
silver_to_gold,gold_contributor_activity,2877,0,5
silver_to_gold,agg_repo_daily,0,2795,5
silver_to_gold,fact_pulls,0,5537,7
silver_to_gold,fact_issues,0,1552,8
silver_to_gold,fact_commits,0,2971,9
silver_to_gold,dim_date,0,0,3
silver_to_gold,dim_user,0,0,4
